#### purpose: track ML experiments with MLflow

In [8]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, classification_report)
import joblib
import os

In [9]:
#load data
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

In [10]:
#set tracking URI to project root
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

#set experiment name
mlflow.set_experiment("titanic-survival") #create experiment if doesnt exist

print("MLflow ready")
print(f"Tracking URI: {mlflow.get_tracking_uri()}") #shows where mlflow saves data

MLflow ready
Tracking URI: sqlite:///../mlflow.db


In [11]:
#run 1- Logistic Regression
with mlflow.start_run(run_name="LogisticRegression_baseline"):

    #parameters we are using
    params={
        "model_type": "LogisticRegression",
        "max_iter": 1000,
        "random_state": 42,
        "test_size": 0.2
    }

    #log all parameters
    mlflow.log_params(params)

    #train model
    lr= LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_train, y_train)
    y_pred = lr.predict(X_test)

    #calculate matrics
    acc = accuracy_score(y_test,y_pred)
    f1_died = f1_score(y_test, y_pred, pos_label=0)
    f1_survived = f1_score(y_test,y_pred, pos_label=1)

    #log metrics
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_died", f1_died)
    mlflow.log_metric("f1_survived",f1_survived)

    #save model
    mlflow.sklearn.log_model(lr,"model")

    print("Run logged")
    print(f"   Accuracy: {acc*100:.2f}%")
    print(f"   F1 Died: {f1_died:.4f}")
    print(f"   F1 Survived: {f1_survived:.4f}")

2026/05/30 12:32:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/30 12:32:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run logged
   Accuracy: 81.01%
   F1 Died: 0.8482
   F1 Survived: 0.7463


In [12]:
#run 2: Random Forest v1
with mlflow.start_run(run_name="RandomForest_v1"):
    params = {
        "model_type": "RandomForestClassifier",
        "n_estimators": 100,
        "max_depth": 5,
        "random_state": 42
    }
    mlflow.log_params(params)

    rf1 = RandomForestClassifier(n_estimators=100,random_state=42,max_depth=5)
    rf1.fit(X_train,y_train)
    y_pred_rf1 = rf1.predict(X_test)

    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred_rf1))
    mlflow.log_metric("f1_died", f1_score(y_test, y_pred_rf1, pos_label=0))
    mlflow.log_metric("f1_survived", f1_score(y_test, y_pred_rf1, pos_label=1))
    mlflow.sklearn.log_model(rf1, "model")

    print(f"RF v1 logged! Accuracy: {accuracy_score(y_test, y_pred_rf1)*100:.2f}%")

2026/05/30 12:32:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/30 12:32:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RF v1 logged! Accuracy: 78.21%


In [13]:
#run 3: Random Forest v2
with mlflow.start_run(run_name="RandomForest_v2"):
    
    params = {
        "model_type": "RandomForestClassifier",
        "n_estimators": 200,
        "max_depth": 8,
        "min_samples_split": 5,
        "min_samples_leaf": 2,
        "class_weight": "balanced",
        "random_state": 42
    }
    mlflow.log_params(params)
    
    rf2 = RandomForestClassifier(n_estimators=200,
                                  max_depth=8,
                                  min_samples_split=5,
                                  min_samples_leaf=2,
                                  class_weight='balanced',
                                  random_state=42)
    rf2.fit(X_train, y_train)
    y_pred_rf2 = rf2.predict(X_test)
    
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred_rf2))
    mlflow.log_metric("f1_died", f1_score(y_test, y_pred_rf2, pos_label=0))
    mlflow.log_metric("f1_survived", f1_score(y_test, y_pred_rf2, pos_label=1))
    mlflow.sklearn.log_model(rf2, "model")

    print(f"RF v2 logged! Accuracy: {accuracy_score(y_test, y_pred_rf2)*100:.2f}%")

2026/05/30 12:32:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/30 12:32:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RF v2 logged! Accuracy: 80.45%
